# Qualité, indicateurs et visualisations

Ce notebook illustre les capacités visuelles et analytiques de `xyt_gps` sur le test set anonymisé : participation, carte GPS, indicateurs par mode et motifs journaliers. Il sert de vitrine rapide pour comprendre ce que le package produit au-delà des tables.

## 1. Préparer l'environnement

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import display

# En développement local, on importe le code source du dépôt.
# Après publication, `pip install xyt-gps` suffira et ce bloc n'aura aucun effet.
for root in (Path.cwd(), *Path.cwd().parents):
    candidates = [root / "package-xyt-gps" / "src", root / "src"]
    for candidate in candidates:
        if (candidate / "xyt_gps").exists() and str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))

import xyt_gps as xyt


def find_demo_data_dir() -> Path:
    """Localiser le dossier anonymous-test-set-gps."""

    env_path = os.environ.get("XYT_GPS_DEMO_DATA_DIR")
    candidates = []
    if env_path:
        candidates.append(Path(env_path).expanduser())
    for root in (Path.cwd(), *Path.cwd().parents):
        candidates.extend([
            root / "Data" / "Output" / "anonymous-test-set-gps",
            root / "anonymous-test-set-gps",
        ])
    for candidate in candidates:
        if (candidate / "storyline.parquet").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Jeu de démonstration introuvable. "
        "Définir XYT_GPS_DEMO_DATA_DIR ou demander le dossier anonymous-test-set-gps à contact@situee.ch."
    )


def example_output_dir(name: str) -> Path:
    """Créer un dossier de sortie local pour les exports et figures du notebook."""

    for root in (Path.cwd(), *Path.cwd().parents):
        if (root / "package-xyt-gps" / "examples").exists():
            path = root / "package-xyt-gps" / "examples" / "_outputs" / name
            path.mkdir(parents=True, exist_ok=True)
            return path
        if (root / "examples").exists() and (root / "src" / "xyt_gps").exists():
            path = root / "examples" / "_outputs" / name
            path.mkdir(parents=True, exist_ok=True)
            return path
    path = Path.cwd() / "xyt_gps_example_outputs" / name
    path.mkdir(parents=True, exist_ok=True)
    return path


DATA_DIR = find_demo_data_dir()
OUTPUT_DIR = example_output_dir("quality-visualisations")

pd.Series({"data_dir": DATA_DIR.name, "output_dir": f"examples/_outputs/{OUTPUT_DIR.name}"}, name="environnement")

## 2. Charger et préparer les données

Le notebook reprend le chemin court pour rester autonome. Les mêmes cellules peuvent être copiées dans un projet réel, en remplaçant seulement le dossier de données et la configuration.

In [ ]:
tables = {
    "storyline": pd.read_parquet(DATA_DIR / "storyline.parquet"),
    "trips": pd.read_parquet(DATA_DIR / "trips.parquet"),
    "journeys": pd.read_parquet(DATA_DIR / "journeys.parquet"),
    "user_statistics": pd.read_parquet(DATA_DIR / "user_statistics.parquet"),
    "sociodemographics": pd.read_parquet(DATA_DIR / "sociodemographics.parquet"),
}
manifest = json.loads((DATA_DIR / "test_set_manifest.json").read_text(encoding="utf-8"))

volume = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(table),
            "columns": len(table.columns),
            "users": table["user_id"].nunique() if "user_id" in table.columns else None,
        }
        for name, table in tables.items()
    ]
)

display(pd.Series(manifest, name="manifest"))
display(volume)
phases = (
    xyt.Phase("Phase1", "2025-04-14", "2025-05-04"),
    xyt.Phase("Phase2", "2025-05-05", "2025-06-01"),
    xyt.Phase("Phase3", "2025-06-02", "2025-06-15"),
)

config = xyt.ProjectConfig(
    experiment_name="anonymous-test-set-gps",
    motiontag_project_name="test-project-gps",
    start_expe="2025-04-14",
    end_expe="2025-06-15",
    phases=phases,
    reference_year=2025,
)

raw = xyt.RawGpsData(
    storyline=tables["storyline"],
    trips=tables["trips"],
    journeys=tables["journeys"],
    user_statistics=tables["user_statistics"],
)

schema_report = xyt.check_raw_import_columns(
    raw.storyline,
    raw.user_statistics,
    trips=raw.trips,
    journeys=raw.journeys,
    raise_on_error=False,
)

schema_report.groupby(["table", "status"], dropna=False).size().rename("columns").reset_index()
dataset = xyt.prepare_mobility_dataset(
    raw,
    config,
    sociodemo=tables["sociodemographics"],
    resample_missing_days=True,
    clean_leg_geometries=True,
    add_length_outlier_flags=True,
    add_signal_quality_flags=True,
)

mobility_tables = xyt.mobility_dataset_tables(dataset)
summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(table),
            "columns": len(table.columns),
            "users": table["user_id"].nunique() if "user_id" in table.columns else None,
        }
        for name, table in mobility_tables.items()
    ]
)

display(summary)
display(dataset.legs[["user_id", "started_at", "finished_at", "mode_niv1", "phase", "length"]].head())

## 3. Participation et qualité de suivi

La grille de participation rend visibles les semaines suivies par utilisateur. Le rapport qualité donne un résumé compact des volumes et de la complétude.

In [ ]:
participation_grid = xyt.build_weekly_participation_grid(dataset.storyline, config)
participation_summary = xyt.summarize_participation_grid(participation_grid)
tracking_quality = xyt.build_tracking_quality_report(dataset.user_stats)

heatmap = xyt.plot_participation_heatmap(
    participation_grid,
    summary=participation_summary,
    notes=[
        "Test set anonymisé : 12 utilisateurs.",
        "Les jours resamplés sont utiles pour tester les exports, pas pour une interprétation territoriale.",
    ],
    max_users=12,
    save_path=OUTPUT_DIR / "participation_heatmap.html",
)

display(tracking_quality)
display(participation_summary.head())
heatmap

## 4. Carte GPS de contrôle

La carte sert à vérifier rapidement que les géométries se lisent, que les modes sont cohérents et que l'échantillon reste plausible. Elle ne doit pas être interprétée comme une carte de diagnostic territorial.

In [ ]:
map_users = sorted(dataset.legs["user_id"].dropna().astype(str).unique())[:3]
gps_map = xyt.plot_gps_traces(
    dataset,
    user_ids=map_users,
    sample_n=120,
    color_by="mode_niv1",
    show_staypoints=True,
    use_antpath=False,
    save_path=OUTPUT_DIR / "gps_trace_map.html",
)

gps_map

## 5. Indicateurs par mode et profil de demande

Les indicateurs agrègent les distances, temps de trajet et volumes de déplacements. Le profil de demande ajoute une lecture horaire des déplacements observés.

In [ ]:
indicators = xyt.compute_mobility_indicators(dataset, config=config)
indicators.metadata["demand_profile"] = xyt.build_daily_demand_profile(dataset.legs, indicators.person_day)

indicator_fig = xyt.plot_indicator_bars(
    indicators,
    metrics=["distance_km_per_day", "travel_time_min_per_day", "trip_count_per_day"],
    max_bars=6,
    title="Indicateurs de mobilité du test set anonymisé",
    save_path=OUTPUT_DIR / "indicator_bars.html",
)

display(indicators.population.head(12))
indicator_fig

## 6. Motifs journaliers de mobilité

Les motifs décrivent la structure des enchaînements de lieux au cours d'une journée. Sur ce test set, ils servent surtout à démontrer la mécanique de construction et de visualisation.

In [ ]:
motifs = xyt.build_mobility_motifs(dataset)
motif_summary = xyt.summarize_mobility_motifs(motifs)
motif_fig = xyt.plot_mobility_motif_graphs(
    motifs,
    top_n=4,
    title="Motifs journaliers du test set anonymisé",
    save_path=OUTPUT_DIR / "mobility_motifs.html",
)

display(motif_summary.head(10))
motif_fig

## 7. Fichiers produits

Les figures HTML sont écrites dans `OUTPUT_DIR`. Ce dossier est local et peut être supprimé ou régénéré sans conséquence.

In [ ]:
sorted(path.name for path in OUTPUT_DIR.glob("*.html"))